# 🚄 Railway Complaint AI Training - Enhanced (90%+ Accuracy)

**⚡ GPU Training** - Completes in ~20-30 minutes (vs 2-3 hours on CPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

## 📋 What this notebook does:
- Trains 4 AI classifiers: **Category**, **Staff**, **Priority**, **Severity**
- Uses advanced techniques: **Focal Loss**, **Class Weighting**, **Early Stopping**
- Target: **90%+ accuracy** on all models
- Automatically saves trained models

---

## 🔧 Setup Instructions:

### ✅ **RECOMMENDED: Use Web Colab**

1. Go to: **https://colab.research.google.com**
2. Upload this notebook (`Train_Enhanced_Models_Colab.ipynb`)
3. **Runtime** → Change runtime type → **GPU (T4)**
4. **Files panel** (left 📁) → Upload `Train_Enhanced_Models_Colab_Dataset.csv`
5. **Runtime** → Run all cells
6. Download trained models after ~25 minutes

---

### ⚠️ **VS Code Colab Extension (Not Recommended)**

The VS Code extension has file upload issues. If you insist on using it:

1. Click **Select Kernel** (top-right) → **Colab** → **New Colab Server**
2. Sign in with Google when prompted
3. Select **GPU runtime** (T4 recommended)
4. Use web Colab's file upload instead (see above)

---

## ⏱️ Expected Training Time:
- **GPU (T4):** 20-30 minutes total ⚡
- **CPU:** 2-3 hours (not recommended) 🐌

## 📊 Expected Results:
- **Category:** 85-90% Macro-F1
- **Staff:** 90-95% Macro-F1  
- **Priority:** 90-95% Macro-F1
- **Severity:** 88-92% Macro-F1

## 📦 Step 1: Install Dependencies

In [1]:
!pip install transformers==4.57.3 torch torchvision torchaudio scikit-learn pandas numpy tqdm -q

import torch
print(f"✅ PyTorch {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

✅ PyTorch 2.9.0+cu126
✅ CUDA Available: True
✅ GPU: Tesla T4


### ⚠️ Important: Verify GPU is Enabled

If you see **CUDA Available: False**, you need to:
1. **Sign out:** Press `Ctrl+Shift+P` → Type "Colab: Sign Out"
2. **Reconnect with GPU:** Click kernel picker → Colab → New Colab Server
3. **Select GPU runtime** when prompted (T4 recommended)
4. **Re-run** the cell above

**Expected output:**
```
✅ CUDA Available: True
✅ GPU: Tesla T4
```

## 📂 Step 2: Load Dataset

The notebook will automatically find `Train_Enhanced_Models_Colab_Dataset.csv` in the same directory.

**If file is not found**, you'll be prompted to upload it via drag-and-drop.

### ⚠️ **VS Code Colab Extension Limitation**

The `files.upload()` widget **does not work** in VS Code Colab extension.

### ✅ **SOLUTION: Use Web Colab (Recommended)**

1. Go to: **https://colab.research.google.com**
2. **Upload this notebook**: `Train_Enhanced_Models_Colab.ipynb`
3. **Runtime** → Change runtime type → **GPU (T4)**
4. **Files panel** (left sidebar 📁) → Click Upload button
5. Select: `Train_Enhanced_Models_Colab_Dataset.csv`
6. **Run all cells**

**Why web Colab is better:**
- ✅ File upload widget works perfectly
- ✅ Better GPU runtime management
- ✅ Easier file management
- ✅ Same free T4 GPU access
- ✅ Training completes in ~25 minutes

---

### 🔄 Alternative: Continue in VS Code (Advanced)

If you must use VS Code, skip to **Step 2B** below for a workaround.

In [2]:
import pandas as pd
import os

print("📂 Loading dataset...\n")

# Try to load from local Windows path first (works in VS Code)
local_path = r"D:\Projects\Rail_Madad\Train_Enhanced_Models_Colab_Dataset.csv"

try:
    # Attempt to read from local path
    print(f"   Trying local path: {local_path}")
    df = pd.read_csv(local_path)
    print(f"   ✅ Success! Loaded from local filesystem")
    dataset_path = local_path
    
except FileNotFoundError:
    print(f"   ❌ Local path not accessible from Colab server")
    print()
    
    # Try Colab server paths
    colab_paths = [
        "/content/Train_Enhanced_Models_Colab_Dataset.csv",
        "/content/Railway_Complaints_Final_Validated.csv",
        "/content/dataset.csv"
    ]
    
    dataset_path = None
    for path in colab_paths:
        if os.path.exists(path):
            dataset_path = path
            print(f"   ✅ Found on Colab server: {path}")
            df = pd.read_csv(dataset_path)
            break
    
    if dataset_path is None:
        print("   📤 No dataset found. Please upload...")
        print()
        print("   🌐 **WEB COLAB:** File picker will appear below")
        print("   💻 **VS CODE:** Upload via Files panel instead")
        print()
        
        from google.colab import files
        uploaded = files.upload()
        
        if uploaded:
            dataset_path = list(uploaded.keys())[0]
            print(f"\n   ✅ Uploaded: {dataset_path}")
            df = pd.read_csv(dataset_path)
        else:
            raise FileNotFoundError(
                "\n❌ SOLUTION:\n"
                "   1. Go to https://colab.research.google.com\n"
                "   2. Upload this notebook\n"
                "   3. Use Files panel to upload dataset\n"
                "   4. Run all cells"
            )

# Display dataset info
print(f"\n{'='*70}")
print(f"  ✅ DATASET LOADED SUCCESSFULLY")
print(f"{'='*70}")

print(f"\n📊 Dataset Statistics:")
print(f"   • File: {os.path.basename(dataset_path)}")
print(f"   • Total samples: {len(df):,}")
print(f"   • Features: {len(df.columns)}")
print(f"   • Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\n📝 Sample Complaint:")
sample = df['Complaint Description'].iloc[0]
print(f"   \"{sample[:120]}{'...' if len(sample) > 120 else ''}\"")

print(f"\n🎯 Classification Tasks:")
print(f"   • Categories: {df['Category'].nunique()} classes")
print(f"   • Staff types: {df['Staff Assignment'].nunique()} classes") 
print(f"   • Priority levels: {df['Auto Priority'].nunique()} classes")
print(f"   • Severity levels: {df['Auto Severity'].nunique()} classes")

# Check GPU
import torch
gpu_available = torch.cuda.is_available()
device_name = torch.cuda.get_device_name(0) if gpu_available else 'CPU'

print(f"\n🚀 Training Setup:")
print(f"   • Device: {device_name}")
print(f"   • Batch size: {32 if gpu_available else 16}")
print(f"   • Estimated time: {'~25 minutes ⚡' if gpu_available else '2-3 hours 🐌'}")

if not gpu_available:
    print(f"\n   ⚠️  WARNING: No GPU detected!")
    print(f"      Runtime → Change runtime type → GPU (T4)")

print(f"\n{'='*70}")
print(f"  🎯 Ready to train 4 AI models!")

📂 Loading dataset...

   Trying local path: D:\Projects\Rail_Madad\Train_Enhanced_Models_Colab_Dataset.csv
   ❌ Local path not accessible from Colab server

   📤 No dataset found. Please upload...

   🌐 **WEB COLAB:** File picker will appear below
   💻 **VS CODE:** Upload via Files panel instead



KeyboardInterrupt: 

## 🧠 Step 3: Define Enhanced Training Components

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np

# ========== FOCAL LOSS ==========
class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance"""
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# ========== EARLY STOPPING ==========
class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=3, min_delta=0.001, mode='max'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        
    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
            return False
            
        if self.mode == 'max':
            improved = score > self.best_score + self.min_delta
        else:
            improved = score < self.best_score - self.min_delta
            
        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                
        return self.early_stop

# ========== LAYER UNFREEZER ==========
class LayerUnfreezer:
    """Gradually unfreeze transformer layers"""
    def __init__(self, model, unfreeze_schedule=[2, 4, 6]):
        self.model = model
        self.unfreeze_schedule = unfreeze_schedule
        self.current_epoch = 0
        
    def step(self, epoch):
        self.current_epoch = epoch
        if epoch in self.unfreeze_schedule:
            num_layers_to_unfreeze = self.unfreeze_schedule.index(epoch) + 1
            self._unfreeze_layers(num_layers_to_unfreeze)
            
    def _unfreeze_layers(self, num_layers):
        layers = self.model.distilbert.transformer.layer
        total_layers = len(layers)
        start_layer = max(0, total_layers - num_layers)
        
        for i in range(start_layer, total_layers):
            for param in layers[i].parameters():
                param.requires_grad = True

# ========== HELPER FUNCTIONS ==========
def compute_class_weights(labels):
    """Compute balanced class weights"""
    unique_labels = np.unique(labels)
    weights = compute_class_weight('balanced', classes=unique_labels, y=labels)
    return torch.FloatTensor(weights)

def compute_metrics(true_labels, pred_labels, label_names=None):
    """Compute comprehensive metrics"""
    accuracy = accuracy_score(true_labels, pred_labels)
    macro_f1 = f1_score(true_labels, pred_labels, average='macro')
    macro_precision = precision_score(true_labels, pred_labels, average='macro', zero_division=0)
    macro_recall = recall_score(true_labels, pred_labels, average='macro', zero_division=0)
    
    per_class_f1 = f1_score(true_labels, pred_labels, average=None, zero_division=0)
    
    metrics = {
        'accuracy': float(accuracy),
        'macro_f1': float(macro_f1),
        'macro_precision': float(macro_precision),
        'macro_recall': float(macro_recall),
        'per_class_f1': per_class_f1.tolist()
    }
    
    if label_names is not None:
        metrics['per_class_f1_named'] = {name: float(f1) for name, f1 in zip(label_names, per_class_f1)}
    
    return metrics

print("✅ Enhanced training components loaded")

## 🎯 Step 4: Training Function

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import pickle
import json
import os

class ComplaintDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }

def train_model(df, target_column, model_name, epochs=10, batch_size=32, focal_gamma=2.0, patience=3):
    """
    Train a single classifier with all enhancements
    
    Args:
        df: DataFrame with complaints
        target_column: Column to predict (Category, Staff Assignment, etc.)
        model_name: Name for saving
        epochs: Number of training epochs
        batch_size: Batch size (use 32 for GPU, 16 for CPU)
        focal_gamma: Gamma parameter for focal loss (higher = more focus on hard examples)
        patience: Early stopping patience (epochs without improvement)
    """
    print(f"\n{'='*70}")
    print(f"  Training: {model_name}")
    print(f"  Target: {target_column}")
    print(f"  Epochs: {epochs}, Focal Gamma: {focal_gamma}, Patience: {patience}")
    print(f"{'='*70}\n")
    
    # Prepare data
    texts = df['Complaint Description'].values
    labels = df[target_column].values
    
    # Encode labels
    label_encoder = LabelEncoder()
    encoded_labels = label_encoder.fit_transform(labels)
    num_classes = len(label_encoder.classes_)
    
    print(f"Classes: {num_classes}")
    print(f"Distribution: {dict(zip(*np.unique(labels, return_counts=True)))}\n")
    
    # Train/Val/Test split
    X_temp, X_test, y_temp, y_test = train_test_split(
        texts, encoded_labels, test_size=0.15, random_state=42, stratify=encoded_labels
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp
    )
    
    print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}\n")
    
    # Tokenizer and model
    tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
    model = DistilBertForSequenceClassification.from_pretrained(
        'distilbert-base-uncased',
        num_labels=num_classes
    )
    
    # Freeze base model initially
    for param in model.distilbert.parameters():
        param.requires_grad = False
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    print(f"Device: {device}\n")
    
    # Datasets
    train_dataset = ComplaintDataset(X_train, y_train, tokenizer)
    val_dataset = ComplaintDataset(X_val, y_val, tokenizer)
    test_dataset = ComplaintDataset(X_test, y_test, tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    
    # Focal Loss with class weights - USE CUSTOM GAMMA
    class_weights = compute_class_weights(y_train).to(device)
    criterion = FocalLoss(alpha=class_weights, gamma=focal_gamma)
    
    # Optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=2e-5)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )
    
    # Training components - USE CUSTOM PATIENCE
    early_stopping = EarlyStopping(patience=patience, mode='max')
    layer_unfreezer = LayerUnfreezer(model, unfreeze_schedule=[2, 4, 6])
    
    best_val_f1 = 0.0
    history = {'train_loss': [], 'val_loss': [], 'val_accuracy': [], 'val_macro_f1': []}
    
    # Training loop
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        
        # Unfreeze layers
        layer_unfreezer.step(epoch)
        
        # Training
        model.train()
        train_loss = 0
        
        for batch in tqdm(train_loader, desc='Training'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc='Validation'):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits, labels)
                val_loss += loss.item()
                
                preds = torch.argmax(outputs.logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        val_metrics = compute_metrics(all_labels, all_preds)
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_accuracy'].append(val_metrics['accuracy'])
        history['val_macro_f1'].append(val_metrics['macro_f1'])
        
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val Loss: {avg_val_loss:.4f}")
        print(f"Val Accuracy: {val_metrics['accuracy']:.4f}")
        print(f"Val Macro-F1: {val_metrics['macro_f1']:.4f}")
        
        # Save best model
        if val_metrics['macro_f1'] > best_val_f1:
            best_val_f1 = val_metrics['macro_f1']
            os.makedirs(f'/content/{model_name}', exist_ok=True)
            model.save_pretrained(f'/content/{model_name}')
            tokenizer.save_pretrained(f'/content/{model_name}')
            with open(f'/content/{model_name}/label_encoder.pkl', 'wb') as f:
                pickle.dump(label_encoder, f)
            print(f"✅ Best model saved (F1: {best_val_f1:.4f})")
        
        # Early stopping
        if early_stopping(val_metrics['macro_f1']):
            print(f"\n⚠️ Early stopping at epoch {epoch+1}")
            break
    
    # Test evaluation
    print(f"\n{'='*70}")
    print("  FINAL TEST EVALUATION")
    print(f"{'='*70}\n")
    
    model.eval()
    test_preds = []
    test_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Testing'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            test_preds.extend(preds.cpu().numpy())
            test_labels.extend(labels.cpu().numpy())
    
    test_metrics = compute_metrics(test_labels, test_preds, label_encoder.classes_)
    
    print(f"\n✅ Test Accuracy: {test_metrics['accuracy']:.4f}")
    print(f"✅ Test Macro-F1: {test_metrics['macro_f1']:.4f}")
    print(f"✅ Test Precision: {test_metrics['macro_precision']:.4f}")
    print(f"✅ Test Recall: {test_metrics['macro_recall']:.4f}\n")
    
    # Save metrics
    with open(f'/content/{model_name}/test_metrics.json', 'w') as f:
        json.dump(test_metrics, f, indent=2)
    with open(f'/content/{model_name}/training_history.json', 'w') as f:
        json.dump(history, f, indent=2)
    
    return test_metrics, model_name

print("✅ Training function ready")

## 🚀 Step 5: Train All 4 Models

This will take ~20-30 minutes on GPU

In [ ]:
import time

start_time = time.time()
results = {}

# 1. Category Classifier (15 classes) - Already achieved 96.15% ✅
print("\n" + "="*70)
print("🎯 Training Category Model (epochs=10, gamma=2.0)")
print("="*70)
metrics_cat, name_cat = train_model(df, 'Category', 'category_model', epochs=10, batch_size=32)
results['Category'] = metrics_cat

# 2. Staff Assignment (6 classes) - Already achieved 93.06% ✅
print("\n" + "="*70)
print("🎯 Training Staff Model (epochs=10, gamma=2.0)")
print("="*70)
metrics_staff, name_staff = train_model(df, 'Staff Assignment', 'staff_model', epochs=10, batch_size=32)
results['Staff'] = metrics_staff

# 3. Priority (3 classes) - ENHANCED: More epochs, higher focal gamma, longer patience
print("\n" + "="*70)
print("⚡ Training Priority Model - ENHANCED (epochs=15, gamma=3.0, patience=5)")
print("="*70)
metrics_priority, name_priority = train_model(
    df, 'Auto Priority', 'priority_model', 
    epochs=15,  # Increased from 10
    batch_size=32,
    focal_gamma=3.0,  # Increased from 2.0
    patience=5  # Increased from 3
)
results['Priority'] = metrics_priority

# 4. Severity (4 classes) - ENHANCED: More epochs, higher focal gamma, longer patience
print("\n" + "="*70)
print("⚡ Training Severity Model - ENHANCED (epochs=15, gamma=3.0, patience=5)")
print("="*70)
metrics_severity, name_severity = train_model(
    df, 'Auto Severity', 'severity_model', 
    epochs=15,  # Increased from 10
    batch_size=32,
    focal_gamma=3.0,  # Increased from 2.0
    patience=5  # Increased from 3
)
results['Severity'] = metrics_severity

elapsed = (time.time() - start_time) / 60

print("\n" + "="*70)
print("  🎉 ALL MODELS TRAINED SUCCESSFULLY!")
print("="*70)
print(f"\nTotal time: {elapsed:.1f} minutes\n")

print("\n📊 FINAL RESULTS:\n")
for name, metrics in results.items():
    print(f"{name}:")
    print(f"  Accuracy: {metrics['accuracy']:.2%}")
    print(f"  Macro-F1: {metrics['macro_f1']:.2%}")
    status = "✅ TARGET ACHIEVED" if metrics['macro_f1'] >= 0.90 else "⚠️ BELOW TARGET"
    print(f"  {status}\n")

## 📥 Step 6: Download Trained Models

Download all 4 models as a ZIP file

In [ ]:
!zip -r trained_models.zip /content/category_model /content/staff_model /content/priority_model /content/severity_model

from google.colab import files
files.download('trained_models.zip')

print("\n✅ Download complete!")
print("\nExtract on your local machine to:")
print("  backend/ai_models/models/enhanced/")
print("\nThen update your Django service to use these models.")